In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Load the dataset
df = pd.read_csv("DATASET-balanced.csv")

# 2. Encode the 'LABEL' column (FAKE → 0, REAL → 1)
df['LABEL'] = df['LABEL'].map({'FAKE': 0, 'REAL': 1})

# 3. Split features (X) and labels (y)
X = df.drop('LABEL', axis=1)
y = df['LABEL']

# 4. Normalize the feature values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Reshape for CNN in next step
# (11778, 26) → (11778, 26, 1) since CNN expects 2D input per sample
X_train_cnn = X_train.reshape(-1, X.shape[1], 1)
X_test_cnn = X_test.reshape(-1, X.shape[1], 1)

print("✅ Data Preprocessing Done.")
print("X_train shape:", X_train_cnn.shape)
print("y_train shape:", y_train.shape)


✅ Data Preprocessing Done.
X_train shape: (9422, 26, 1)
y_train shape: (9422,)


In [18]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

# Define the CNN model
model = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),

    Conv1D(128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)                    │ (None, 24, 64)              │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_2 (MaxPooling1D)       │ (None, 12, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_3 (Conv1D)                    │ (None, 10, 128)             │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_3 (MaxPooling1D)       │ (None, 5, 128)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 640)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │          82,048 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 107,137 (418.50 KB)

 Trainable params: 107,137 (418.50 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
# Train the CNN model
history = model.fit(
    X_train_cnn, y_train,
    validation_data=(X_test_cnn, y_test),
    epochs=15,
    batch_size=32
)


Epoch 1/15
295/295 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.7968 - loss: 0.4205 - val_accuracy: 0.9571 - val_loss: 0.1198
Epoch 2/15
295/295 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9660 - loss: 0.1120 - val_accuracy: 0.9817 - val_loss: 0.0677
Epoch 3/15
295/295 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9807 - loss: 0.0630 - val_accuracy: 0.9847 - val_loss: 0.0545
Epoch 4/15
295/295 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9840 - loss: 0.0533 - val_accuracy: 0.9907 - val_loss: 0.0353
Epoch 5/15
295/295 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9866 - loss: 0.0366 - val_accuracy: 0.9890 - val_loss: 0.0368
Epoch 6/15
295/295 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9866 - loss: 0.0354 - val_accuracy: 0.9847 - val_loss: 0.0450
Epoch 7/15
295/295 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9877 - loss: 0.0345 - val_accuracy: 0.9907 - val_loss: 0.0263
Epoch 8/15
295/295 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9928 - loss: 0.0202 - val_accuracy: 

In [20]:
import joblib

# Save model
model.save("cnn_deepfake_audio_model.h5")

# Save scaler
joblib.dump(scaler, "scaler.pkl")


['scaler.pkl']

In [21]:
# Evaluate the model
loss, accuracy = model.evaluate(X_test_cnn, y_test)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")


74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9853 - loss: 0.0483
Test Loss: 0.0471
Test Accuracy: 98.73%


In [22]:
import librosa
import numpy as np

def extract_features(audio_path):
    y, sr = librosa.load(audio_path, sr=22050, duration=4)
    y = librosa.util.fix_length(y, size=sr*4)

    features = [
        librosa.feature.chroma_stft(y=y, sr=sr).mean(),
        librosa.feature.rms(y=y).mean(),
        librosa.feature.spectral_centroid(y=y, sr=sr).mean(),
        librosa.feature.spectral_bandwidth(y=y, sr=sr).mean(),
        librosa.feature.spectral_rolloff(y=y, sr=sr).mean(),
        librosa.feature.zero_crossing_rate(y).mean()
    ]

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    features.extend(mfcc.mean(axis=1))

    return np.array(features)


In [23]:
from tensorflow.keras.models import load_model
import joblib

# Load the saved model and scaler
model = load_model("cnn_deepfake_audio_model.h5")
scaler = joblib.load("scaler.pkl")

# Extract features from new audio
audio_path = "taylor-original.wav"  # Replace with your path
features = extract_features(audio_path)

# Scale and reshape
scaled = scaler.transform([features])
reshaped = scaled.reshape(1, -1, 1)

# Predict
prediction = model.predict(reshaped)[0][0]
label = "REAL" if prediction >= 0.5 else "FAKE"

print(f"Prediction: {prediction}")
print(f"Predicted Label: {label}")


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
Prediction: 0.9892411828041077
Predicted Label: REAL
